# 第一阶段 MySQL 实测核验

文件：`part1_mysql/verification.ipynb`。环境由父聊天配置，本次只读复用；本 Notebook 不写入房源。

In [1]:
import json
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from IPython.display import display
import pandas as pd
from check_mysql import connect_mysql

environment = json.loads(Path("evidence/environment.json").read_text(encoding="utf-8"))
print("Environment evidence:", environment["checked_at"])
print("Docker Engine:", environment["docker_engine_version"])
for label in ("workspace", "mysql"):
    item = environment[label]
    print(item["name"], "running=", item["running"], "health=", item["health"])
print("Jupyter HTTP:", environment["jupyter"]["status_code"])
print("MySQL published ports:", environment["mysql"]["published_ports"])
print("L05 volume:", environment["volume"]["name"])
for mount in environment["workspace"]["mounts"]:
    if mount["Destination"] == "/home/pwuser/workspace":
        print("Bind:", mount["Source"], "->", mount["Destination"])
for line in environment["teaching_helper"]["output"]:
    print(line)
print("environment_ok:", environment["environment_ok"])

Environment evidence: 2026-10-27T15:30:31.3838545+08:00
Docker Engine: 29.8.0
smartpark-student-course-workspace-1 running= True health= healthy
smartpark-student-mysql-1 running= True health= healthy
Jupyter HTTP: 200
MySQL published ports: {'3306/tcp': None, '33060/tcp': None}
L05 volume: smartpark-student_l05-mysql-data
Bind: D:\大数据采集与处理-学生实验环境\workspace -> /home/pwuser/workspace
[PASS] Docker课程MySQL保护检查通过；Windows本机MySQL不会被操作。
[PASS] 仅检查完成，没有启动、停止或修改服务。
environment_ok: True


In [2]:
def show_query(sql):
    print("SQL:", sql)
    with connect_mysql() as connection:
        with connection.cursor() as cursor:
            cursor.execute(sql)
            if cursor.description:
                display(pd.DataFrame(cursor.fetchall(), columns=[item[0] for item in cursor.description]))
            else:
                print("OK")

print("Live MySQL query:", datetime.now(ZoneInfo("Asia/Shanghai")).isoformat(timespec="seconds"))
show_query("SELECT VERSION() AS version, DATABASE() AS database_name, CURRENT_USER() AS account_name")
show_query("SHOW DATABASES")
show_query("USE smartpark_lab")
show_query("SHOW TABLES")

Live MySQL query: 2026-10-27T15:33:17+08:00
SQL: SELECT VERSION() AS version, DATABASE() AS database_name, CURRENT_USER() AS account_name


,version,database_name,account_name
0,8.4.11,smartpark_lab,student@%


SQL: SHOW DATABASES


,Database
0,information_schema
1,performance_schema
2,smartpark_lab


SQL: USE smartpark_lab
OK
SQL: SHOW TABLES


,Tables_in_smartpark_lab
0,houseinfo


In [3]:
print("Live table check:", datetime.now(ZoneInfo("Asia/Shanghai")).isoformat(timespec="seconds"))
show_query("DESCRIBE houseinfo")
show_query("SELECT COUNT(*) AS before_count FROM houseinfo")
show_query("SELECT TABLE_COLLATION, ENGINE FROM information_schema.TABLES WHERE TABLE_SCHEMA='smartpark_lab' AND TABLE_NAME='houseinfo'")

Live table check: 2026-10-27T15:33:17+08:00
SQL: DESCRIBE houseinfo


,Field,Type,Null,Key,Default,Extra
0,id,int,NO,PRI,None,auto_increment
1,title,varchar(200),YES,,None,
2,house,varchar(50),YES,,None,
3,address,varchar(200),YES,,None,
4,struct,varchar(50),YES,,None,
5,area,varchar(20),YES,,None,
6,floor,varchar(20),YES,,None,
7,maketime,varchar(20),YES,,None,
8,price,varchar(20),YES,,None,


SQL: SELECT COUNT(*) AS before_count FROM houseinfo


,before_count
0,0


SQL: SELECT TABLE_COLLATION, ENGINE FROM information_schema.TABLES WHERE TABLE_SCHEMA='smartpark_lab' AND TABLE_NAME='houseinfo'


,TABLE_COLLATION,ENGINE
0,utf8mb4_0900_ai_ci,InnoDB


In [4]:
# 独立重跑安全建表检查：已有表时只查询，保存独立证据。
import contextlib
import io
from check_mysql import run
with contextlib.redirect_stdout(io.StringIO()):
    repeat = run()
print("Recheck:", repeat["checked_at"])
for key in ("state", "version", "current_user", "table_existed_before", "created_table", "schema_ok", "before_count", "record_writes"):
    print(key, "=", repeat.get(key))
assert repeat["state"] == "ready"
assert repeat["created_table"] is False
assert repeat["record_writes"] == 0
print("Existing-table recheck passed; no house record was written.")

Recheck: 2026-10-27T15:33:17+08:00
state = ready
version = 8.4.11
current_user = student@%
table_existed_before = True
created_table = False
schema_ok = True
before_count = 0
record_writes = 0
Existing-table recheck passed; no house record was written.
